In [192]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/aith-dl-competition-tabular-data/sample_submission.csv
/kaggle/input/aith-dl-competition-tabular-data/train.csv
/kaggle/input/aith-dl-competition-tabular-data/test.csv


In [193]:
!pip install pytorch_lightning -q

In [194]:
!pip install -q pyngrok

In [195]:
import pytorch_lightning as pl
from torchmetrics.classification import BinaryAUROC, BinaryF1Score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchmetrics import MetricCollection
from torch.utils.data import random_split, Subset, DataLoader, TensorDataset
import pandas as pd
from pathlib import Path
from pytorch_lightning.callbacks import TQDMProgressBar, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

In [ ]:
pl.seed_everything(42)

In [196]:
df_time = pd.read_csv('/kaggle/input/aith-dl-competition-tabular-data/train.csv')
df_time.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   15000 non-null  int64  
 1   age                  15000 non-null  float64
 2   height(cm)           15000 non-null  float64
 3   weight(kg)           15000 non-null  float64
 4   waist(cm)            15000 non-null  float64
 5   eyesight(left)       15000 non-null  float64
 6   eyesight(right)      15000 non-null  float64
 7   hearing(left)        15000 non-null  float64
 8   hearing(right)       15000 non-null  float64
 9   systolic             15000 non-null  float64
 10  relaxation           15000 non-null  float64
 11  fasting blood sugar  15000 non-null  float64
 12  Cholesterol          15000 non-null  float64
 13  triglyceride         15000 non-null  float64
 14  HDL                  15000 non-null  float64
 15  LDL                  15000 non-null 

In [197]:
class TabularTask(pl.LightningDataModule):
    def __init__(self,
                 data_dir = Path('/kaggle/input/aith-dl-competition-tabular-data'),
                 batch_size=2048):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size

    def prepare_data(self):
        self.df_train = pd.read_csv(self.data_dir / 'train.csv').drop(columns = ['id'])
        self.df_test = pd.read_csv(self.data_dir / 'test.csv').drop(columns = ['id'])

    def setup(self, stage:str):
        if stage in ('train', 'fit'):
            X = self.df_train.drop(columns = ['smoking']).values
            y = self.df_train['smoking'].values
            dataset = TensorDataset(torch.tensor(X, dtype = torch.float32),
                                   torch.tensor(y, dtype=torch.float32))
            train_size = int(0.6 * len(dataset))
            val_size = int(0.2 * len(dataset))
            test_size = len(dataset) - train_size - val_size
            self.train_set, self.val_set, self.internal_test_set = random_split(
            dataset, [train_size, val_size, test_size]
        )
            
        if stage == 'test':
            self.test_val = self.internal_test_set

    def train_dataloader(self):
        return DataLoader(self.train_set, batch_size = self.batch_size, shuffle = True)

    def val_dataloader(self):
        return DataLoader(self.val_set, batch_size = self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_val, batch_size = self.batch_size)

In [198]:
class TabularModel(pl.LightningModule):
    def __init__(self, list_of_hidden_layers:int, 
                batch_norm = False, 
                dropout = False,
                only_first_batch_norm = False):
        super().__init__()
        self.hidden_layers = list_of_hidden_layers
        self.layers = list()
        self.batch_norm = batch_norm
        self.dropout = dropout
        self.first_batch_norm = only_first_batch_norm
        
        for index, i in enumerate(self.hidden_layers):
            self.layers.append(nn.Linear(i[0], i[1]))
            if index == 0 and self.first_batch_norm:
                self.layers.append(nn.BatchNorm1d(i[1]))
            elif batch_norm:
                self.layers.append(nn.BatchNorm1d(i[1]))
            elif dropout:
                self.layers.append(nn.Dropout())
            elif i[1] != 1:
                self.layers.append(nn.ReLU()) 

        self.layers.append(nn.Sigmoid())
        self.layers = nn.Sequential(*self.layers)

        self.metrics = MetricCollection([
        BinaryAUROC(), 
        BinaryF1Score()
            ])
    
        self.val_metric = self.metrics.clone(prefix = 'val')
        self.test_metric = self.metrics.clone(prefix = 'test')
    
    def forward(self, x):
        x = self.layers(x)
        return x.squeeze(1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self.forward(x)
        loss = F.binary_cross_entropy(logits, y)
        self.log('train_loss', loss, on_epoch = True, on_step = True, prog_bar = True, logger = True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self.forward(x)
        loss = F.binary_cross_entropy(logits, y)
        self.val_metric.update(logits, y)
        self.log('validation_loss', loss, on_epoch = True, on_step = True, prog_bar = True, logger = True)
        return loss
    
    def test_step(self, batch, batch_idx):
        x, y= batch
        logits = self.forward(x)
        self.test_metric.update(logits, y)
        return logits
    
    def on_validation_epoch_end(self):
        self.log_dict(self.val_metric.compute(), prog_bar = True, on_epoch = True)
        self.val_metric.reset()

    def on_test_epoch_end(self) -> None:
        self.log_dict(self.test_metric.compute(), prog_bar=True, on_epoch=True)
        self.test_metric.reset()

    def configure_optimizers(self): 
        return torch.optim.AdamW(self.parameters(), lr=0.01)

In [41]:
trainer = pl.Trainer(
    max_epochs = 30,
    log_every_n_steps = 5,
    callbacks=[TQDMProgressBar(refresh_rate=10)],
    logger=TensorBoardLogger(save_dir='simple_model', name='cifar10', version='model_v0.1_simple_2_epochs')
)

In [42]:
hidden_l = [
    [22, 20], 
    [20, 18], 
    [18, 16], 
    [16, 14], 
    [14,12], 
    [12, 10], 
    [10, 8], 
    [8, 6], 
    [6, 1]
]

In [43]:
datamodule = TabularTask()
model = TabularModel(hidden_l)

AttributeError: module 'torch.nn' has no attribute 'LekyReLU'

In [ ]:
trainer.fit(model, datamodule)

In [ ]:
from pyngrok import ngrok
import os
from tensorboard import program

LOG_DIR = "/kaggle/working/simple_model/"
os.makedirs(LOG_DIR, exist_ok=True)

tb = program.TensorBoard()
tb.configure(argv=[None, "--logdir", LOG_DIR, "--host", "0.0.0.0", "--port", "6006"])
tb_url = ngrok.connect(6006)  # Порт TensorBoard — 6006
print(f"TensorBoard доступен по адресу: {tb_url}")
tb.main()



NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784



TensorBoard доступен по адресу: NgrokTunnel: "https://7b6b-34-29-52-147.ngrok-free.app" -> "http://localhost:6006"


TensorBoard 2.17.1 at http://0.0.0.0:6006/ (Press CTRL+C to quit)


0

In [ ]:
trainer.test(datamodule = datamodule)

In [ ]:
trainer = pl.Trainer(
    max_epochs = 30,
    log_every_n_steps = 5,
    callbacks=[TQDMProgressBar(refresh_rate=10)],
    logger=TensorBoardLogger(save_dir='simple_model', name='cifar10', version='model_v0.2_withdropout_andbatch_norm')
)

datamodule = TabularTask()
model = TabularModel(hidden_l, batch_norm = True, dropout = True)

In [ ]:
trainer.fit(model, datamodule)

In [ ]:
trainer.test(datamodule = datamodule)

In [ ]:
trainer = pl.Trainer(
    max_epochs = 60,
    log_every_n_steps = 5,
    callbacks=[TQDMProgressBar(refresh_rate=10)],
    logger=TensorBoardLogger(save_dir='simple_model', name='cifar10', version='model_v0.2_with_batch_norm')
)

datamodule = TabularTask()
model = TabularModel(hidden_l, batch_norm = True, dropout = True)

In [ ]:
trainer.fit(model, datamodule)

In [ ]:
trainer.test(datamodule = datamodule)

In [ ]:
trainer = pl.Trainer(
    max_epochs = 150,
    log_every_n_steps = 5,
    callbacks=[TQDMProgressBar(refresh_rate=10)],
    logger=TensorBoardLogger(save_dir='simple_model', name='cifar10', version='model_v0.2_simple_model_150_epoch')
)

datamodule = TabularTask()
model = TabularModel(hidden_l, batch_norm = True, dropout = True)

In [ ]:
trainer.fit(model, datamodule)

In [ ]:
trainer.test(datamodule = datamodule)

In [ ]:
trainer = pl.Trainer(
    max_epochs = 150,
    log_every_n_steps = 5,
    callbacks=[TQDMProgressBar(refresh_rate=10)],
    logger=TensorBoardLogger(save_dir='simple_model', name='cifar10', version='model_v0.2_with_batch_norm_and_dropout_150_epoch')
)

datamodule = TabularTask()
model = TabularModel(hidden_l, batch_norm = True, dropout = True)

In [ ]:
trainer.fit(model, datamodule)

In [ ]:
trainer.test(datamodule = datamodule)

In [ ]:
trainer = pl.Trainer(
    max_epochs = 150,
    log_every_n_steps = 5,
    callbacks=[TQDMProgressBar(refresh_rate=10)],
    logger=TensorBoardLogger(save_dir='simple_model', name='cifar10', version='model_v0.2_with_batch_norm_on_first_layer_150_epoch')
)

datamodule = TabularTask()
model = TabularModel(hidden_l, only_first_batch_norm = True)

In [ ]:
model

In [ ]:
trainer.fit(model, datamodule)

In [ ]:
trainer.test(datamodule = datamodule)

In [ ]:
class TabularModelSC(TabularModel):
    def __init__(self, list_of_hidden_layers:int, 
                scheduler_class,
                scheduler_kwargs,
                batch_norm = False, 
                dropout = False,
                only_first_batch_norm = False):
        super(TabularModelSC, self).__init__(
           list_of_hidden_layers,
            batch_norm,
            dropout,
            only_first_batch_norm
        )
        self.scheduler_class = scheduler_class
        self.scheduler_kwargs = scheduler_kwargs

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=0.01)
        scheduler = self.scheduler_class(
            optimizer=optimizer,
            **self.scheduler_kwargs
        )

        lr_scheduler = {
            "scheduler": scheduler,
            "interval": "epoch",
            "monitor": "validation_loss",
        }
        return [optimizer], [lr_scheduler]

In [ ]:
model = TabularModelSC(list_of_hidden_layers= hidden_l,
    scheduler_class=torch.optim.lr_scheduler.ExponentialLR,
    scheduler_kwargs={
        'gamma': 0.5,
    },
    only_first_batch_norm = True
)

trainer = pl.Trainer(
    max_epochs=150,
    log_every_n_steps=10,
    fast_dev_run=False,
    callbacks=TQDMProgressBar(refresh_rate=10),
    logger=TensorBoardLogger(save_dir='simple_model', name='cifar10', version='model_v0.2_ExponentialLR')

)
trainer.fit(model=model, datamodule=datamodule)

In [ ]:
trainer.test(datamodule=datamodule)

In [ ]:
model = TabularModelSC(list_of_hidden_layers= hidden_l,
                       scheduler_class=torch.optim.lr_scheduler.CyclicLR,
                       scheduler_kwargs={
                        'step_size_up': 100,
                        'mode': 'triangular',
                        'base_lr': 0.001,
                        'max_lr': 0.01
                        },
                       only_first_batch_norm = True
)

trainer = pl.Trainer(
    max_epochs=150,
    log_every_n_steps=10,
    fast_dev_run=False,
    callbacks=TQDMProgressBar(refresh_rate=10),
    logger=TensorBoardLogger(save_dir='simple_model', name='tabular', version='model_v0.2_CyclicLR_with_1_batch_norm')

)
trainer.fit(model=model, datamodule=datamodule)

In [ ]:
trainer.test(datamodule = datamodule)

In [ ]:
model = TabularModelSC(list_of_hidden_layers= hidden_l,
                       scheduler_class=torch.optim.lr_scheduler.CyclicLR,
                       scheduler_kwargs={
                        'step_size_up': 100,
                        'mode': 'triangular',
                        'base_lr': 0.001,
                        'max_lr': 0.01
                        },
                       only_first_batch_norm = True
)

trainer = pl.Trainer(
    max_epochs=300,
    log_every_n_steps=10,
    fast_dev_run=False,
    callbacks=TQDMProgressBar(refresh_rate=10),
    logger=TensorBoardLogger(save_dir='simple_model', name='tabular', version='model_v0.2_CyclicLR_with_1_batch_norm_300_epoch')

)
trainer.fit(model=model, datamodule=datamodule)

In [ ]:
trainer.test(datamodule = datamodule)

In [ ]:
model = TabularModelSC(list_of_hidden_layers= hidden_l,
                       scheduler_class=torch.optim.lr_scheduler.CyclicLR,
                       scheduler_kwargs={
                        'step_size_up': 100,
                        'mode': 'triangular',
                        'base_lr': 0.001,
                        'max_lr': 0.01
                        },
                       only_first_batch_norm = True
)

trainer = pl.Trainer(
    max_epochs=300,
    log_every_n_steps=10,
    fast_dev_run=False,
    callbacks=[TQDMProgressBar(refresh_rate=10), EarlyStopping(monitor="validation_loss", mode="min", patience=10)],
    logger=TensorBoardLogger(save_dir='simple_model', name='tabular', version='model_v0.2_CyclicLR_with_1_batch_norm_300_epoch_ES_with_Sig')

)
trainer.fit(model=model, datamodule=datamodule)

In [ ]:
trainer.test(datamodule=datamodule)

In [ ]:
test = torch.tensor(pd.read_csv('/kaggle/input/aith-dl-competition-tabular-data/test.csv').drop(columns = ['id']).values).float()
data_test_loader = DataLoader(test, batch_size = 1024)
preds = trainer.predict(model = model, dataloaders = data_test_loader)
all_preds = torch.cat(preds)
all_preds = F.sigmoid(all_preds)

In [ ]:
all_preds.shape

In [ ]:
submission = pd.DataFrame({
    "id": range(15000, len(all_preds) + 15000), 
    "smoking": all_preds.numpy(),
})

submission.to_csv("submission.csv", index = False)

In [ ]:
a = pd.read_csv('/kaggle/working/submission.csv')
a.head()

In [ ]:
y = pd.read_csv('/kaggle/input/aith-dl-competition-tabular-data/test.csv')

In [ ]:
submission.head()

In [52]:
list_of_hidden_layers = [[22, 21], 
                         [21, 20],
                         [20, 19],
                         [19, 19],
                         [19, 18],
                         [17, 16], 
                         [16, 15],
                         [15, 15],
                         [15, 14],
                         [14, 13],
                         [13, 13],
                         [13, 12],
                         [12, 11],
                         [11, 11],
                         [11, 10],
                         [10, 9],
                         [9, 8],
                         [8,7],
                         [7, 6],
                         [6, 5],
                         [5, 5],
                         [5, 3],
                         [3, 2],
                         [2, 1]
                        ]

In [ ]:
model = TabularModelSC(list_of_hidden_layers= hidden_l,
                       scheduler_class=torch.optim.lr_scheduler.CyclicLR,
                       scheduler_kwargs={
                        'step_size_up': 100,
                        'mode': 'triangular',
                        'base_lr': 0.001,
                        'max_lr': 0.01
                        },
                       only_first_batch_norm = True
)

trainer = pl.Trainer(
    max_epochs=100,
    log_every_n_steps=10,
    fast_dev_run=False,
    # callbacks=[TQDMProgressBar(refresh_rate=10), EarlyStopping(monitor="validation_loss", mode="min", patience=10)],
    callbacks = TQDMProgressBar(refresh_rate=10),
    logger=TensorBoardLogger(save_dir='simple_model', name='tabular', version='model_v0.2_CyclicLR_with_1_batch_norm_300_epoch_ES_with_Sig_Another_Arch')

)
trainer.fit(model=model, datamodule=datamodule)

In [ ]:
model = TabularModelSC(list_of_hidden_layers= hidden_l,
                       scheduler_class=torch.optim.lr_scheduler.CyclicLR,
                       scheduler_kwargs={
                        'step_size_up': 100,
                        'mode': 'triangular',
                        'base_lr': 0.001,
                        'max_lr': 0.01
                        },
                       only_first_batch_norm = True
)

trainer = pl.Trainer(
    max_epochs=100,
    log_every_n_steps=10,
    fast_dev_run=False,
    # callbacks=[TQDMProgressBar(refresh_rate=10), EarlyStopping(monitor="validation_loss", mode="min", patience=10)],
    callbacks = TQDMProgressBar(refresh_rate=10),
    logger=TensorBoardLogger(save_dir='simple_model', name='tabular', version='model_v0.5_Simple_decrease_everything')

)
trainer.fit(model=model, datamodule=datamodule)

In [ ]:
trainer.test(datamodule=datamodule)

In [ ]:
model = TabularModelSC(list_of_hidden_layers= hidden_l,
                       scheduler_class=torch.optim.lr_scheduler.ReduceLROnPlateau,
                       scheduler_kwargs={
                           "factor": 0.1,
                           'mode': 'min',
                           'patience': 3,
                       },
                       only_first_batch_norm = True,
                       dropout = True
)

trainer = pl.Trainer(
    max_epochs=300,
    log_every_n_steps=10,
    fast_dev_run=False,
    callbacks=TQDMProgressBar(refresh_rate=10),
    logger=TensorBoardLogger(save_dir='simple_model', name='tabular', version='model_v0.3Reduce_On_plauto')

)
trainer.fit(model=model, datamodule=datamodule)

In [ ]:
trainer.test(datamodule = datamodule)

In [ ]:
test = torch.tensor(pd.read_csv('/kaggle/input/aith-dl-competition-tabular-data/test.csv').drop(columns = ['id']).values).float()
data_test_loader = DataLoader(test, batch_size = 1024)
preds = trainer.predict(model = model, dataloaders = data_test_loader)
all_preds = torch.cat(preds)
all_preds = F.sigmoid(all_preds)

In [ ]:
submission = pd.DataFrame({
    "id": range(15000, len(all_preds) + 15000), 
    "smoking": all_preds.numpy(),
})

submission.to_csv("submission_new_arch.csv", index = False)

In [ ]:
model = TabularModelSC(list_of_hidden_layers= hidden_l,
                       scheduler_class=torch.optim.lr_scheduler.ReduceLROnPlateau,
                       scheduler_kwargs={
                           "factor": 0.1,
                           'mode': 'min',
                           'patience': 3,
                       },
                       only_first_batch_norm = True,
                       dropout = True
)

trainer = pl.Trainer(
    max_epochs=600,
    log_every_n_steps=10,
    fast_dev_run=False,
    callbacks=TQDMProgressBar(refresh_rate=10),
    logger=TensorBoardLogger(save_dir='simple_model', name='tabular', version='model_v0.3Reduce_On_plauto_dropout_600_epoch')

)
trainer.fit(model=model, datamodule=datamodule)

In [ ]:
trainer.test(datamodule = datamodule)

In [141]:
list_of_hidden_layers = [[22, 21],
                         [21, 22],
                         [22, 22],
                         [22, 21],
                         [21, 20],
                         [20, 19],
                         [19, 18],
                         [18, 17],
                         [17, 16], 
                         [16, 15],
                         [15, 16],
                         [16, 15],
                         [15, 14],
                         [14, 13],
                         [13, 12],
                         [12, 11],
                         [11, 10],
                         [10, 9],
                         [9, 8],
                         [8, 7],
                         [7, 6],
                         [6, 5],
                         [5, 4],
                         [4, 3],
                         [3, 4],
                         [4, 3],
                         [3, 2],
                         [2, 1]
                        ]

In [164]:
list_of_hidden_layers = [[22, 15]]
a = [[15, 15]] * 3
for i in a:
    list_of_hidden_layers.append(i)
list_of_hidden_layers.append([15, 1])

In [161]:
list_of_hidden_layers

[[22, 15],
 [15, 15],
 [15, 15],
 [15, 15],
 [15, 15],
 [15, 15],
 [15, 15],
 [15, 15],
 [15, 15],
 [15, 1]]

In [54]:
print(list_of_hidden_layers)

[[22, 21], [21, 22], [22, 22], [22, 21], [21, 20], [20, 19], [19, 18], [18, 17], [17, 16], [16, 15], [15, 16], [16, 15], [15, 14], [14, 13], [13, 12], [12, 11], [11, 10], [10, 9], [9, 8], [8, 7], [7, 6], [6, 5], [5, 4], [4, 3], [3, 4], [4, 3], [3, 2], [2, 1]]


In [ ]:
list_of_hidden_layers = [
                         [22, 256],
                         [256, 128],
                         [128, 64],
                         [64, 32],
                         [32, 1]
                        ]

In [ ]:
model = TabularModelSC(list_of_hidden_layers= hidden_l,
                       scheduler_class=torch.optim.lr_scheduler.CyclicLR,
                       scheduler_kwargs={
                           'step_size_up': 100,
                           'mode': 'exp_range',
                           'base_lr': 0.001,
                           'max_lr': 0.01,
                       },
                       only_first_batch_norm = True
)

trainer = pl.Trainer(
    max_epochs=150,
    log_every_n_steps=10,
    fast_dev_run=False,
    # callbacks= [TQDMProgressBar(refresh_rate=10), EarlyStopping(monitor="validation_loss", mode="min", patience=20)],
    callbacks= TQDMProgressBar(refresh_rate=10),
    logger=TensorBoardLogger(save_dir='simple_model', name='tabular', version='model_v0.4.Simple_decrease_by_columns_CyclicLR_exp_mode_150_epoch')

)
trainer.fit(model=model, datamodule=datamodule)

In [ ]:
model = TabularModelSC(list_of_hidden_layers= hidden_l,
                       scheduler_class=torch.optim.lr_scheduler.CyclicLR,
                       scheduler_kwargs={
                           'step_size_up': 100,
                           'mode': 'exp_range',
                           'base_lr': 0.001,
                           'max_lr': 0.01,
                       },
                       only_first_batch_norm = True
)

trainer = pl.Trainer(
    max_epochs=150,
    log_every_n_steps=10,
    fast_dev_run=False,
    # callbacks= [TQDMProgressBar(refresh_rate=10), EarlyStopping(monitor="validation_loss", mode="min", patience=20)],
    callbacks= TQDMProgressBar(refresh_rate=10),
    logger=TensorBoardLogger(save_dir='simple_model', name='tabular', version='model_v0.4.Simple_decrease_by_columns_CyclicLR_exp_mode_150_epoch')

)
trainer.fit(model=model, datamodule=datamodule)

In [ ]:
model = TabularModelSC(list_of_hidden_layers= hidden_l,
                       scheduler_class=torch.optim.lr_scheduler.CyclicLR,
                       scheduler_kwargs={
                           'step_size_up': 100,
                           'mode': 'exp_range',
                           'base_lr': 0.001,
                           'max_lr': 0.01,
                       },
                       only_first_batch_norm = True,
                       dropout = True
)

trainer = pl.Trainer(
    max_epochs = 100,
    log_every_n_steps=10,
    fast_dev_run=False,
    # callbacks= [TQDMProgressBar(refresh_rate=10), EarlyStopping(monitor="validation_loss", mode="min", patience=20)],
    callbacks= TQDMProgressBar(refresh_rate=10),
    logger=TensorBoardLogger(save_dir='simple_model', name='tabular', version='model_v0.5.20_layers')

)
trainer.fit(model=model, datamodule=datamodule)

In [ ]:
model = TabularModelSC(list_of_hidden_layers= hidden_l,
                       scheduler_class=torch.optim.lr_scheduler.CyclicLR,
                       scheduler_kwargs={
                           'step_size_up': 100,
                           'mode': 'exp_range',
                           'base_lr': 0.01,
                           'max_lr': 0.1,
                       },
                       only_first_batch_norm = True,
                       # dropout = True,
                       batch_norm = True)

trainer = pl.Trainer(
    max_epochs = 100,
    log_every_n_steps=10,
    fast_dev_run=False,
    # callbacks= [TQDMProgressBar(refresh_rate=10), EarlyStopping(monitor="validation_loss", mode="min", patience=20)],
    callbacks= TQDMProgressBar(refresh_rate=10),
    logger=TensorBoardLogger(save_dir='simple_model', name='tabular', version='model_v0.4.Simple_decrease_by_columns_CyclicLR_exp_mode_150_epoch_with_dropout_all_norm')

)
trainer.fit(model=model, datamodule=datamodule)

In [189]:
model = TabularModelSC(list_of_hidden_layers,
                       scheduler_class=torch.optim.lr_scheduler.CyclicLR,
                       scheduler_kwargs={
                           'step_size_up': 100,
                           'mode': 'triangular',
                           'base_lr': 0.001,
                           'max_lr': 0.01
                       },
                       only_first_batch_norm = True
                       # dropout = True,
                       # batch_norm = True
                      )

trainer = pl.Trainer(
    max_epochs = 200,
    log_every_n_steps=10,
    fast_dev_run=False,
    callbacks= [TQDMProgressBar(refresh_rate=10), EarlyStopping(monitor="validation_loss", mode="min", patience=20)],
    # callbacks= TQDMProgressBar(refresh_rate=10),
    logger=TensorBoardLogger(save_dir='simple_model', name='tabular', version='model_v0.5.degree_2_without_dropout')

)

datamodule = TabularTask()

trainer.fit(model=model, datamodule=datamodule)

/usr/local/lib/python3.10/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory simple_model/tabular/model_v0.5.degree_2_without_dropout/checkpoints exists and is not empty.


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/usr/local/lib/python3.10/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/usr/local/lib/python3.10/dist-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [184]:
model

TabularModelSC(
  (layers): Sequential(
    (0): Linear(in_features=22, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Linear(in_features=32, out_features=1, bias=True)
    (9): Sigmoid()
  )
  (metrics): MetricCollection(
    (BinaryAUROC): BinaryAUROC()
    (BinaryF1Score): BinaryF1Score()
  )
  (val_metric): MetricCollection(
    (BinaryAUROC): BinaryAUROC()
    (BinaryF1Score): BinaryF1Score(),
    prefix=val
  )
  (test_metric): MetricCollection(
    (BinaryAUROC): BinaryAUROC()
    (BinaryF1Score): BinaryF1Score(),
    prefix=test
  )
)

In [190]:
trainer.test(datamodule = datamodule)

/usr/local/lib/python3.10/dist-packages/pytorch_lightning/trainer/connectors/checkpoint_connector.py:145: `.test(ckpt_path=None)` was called without a model. The best model of the previous `fit` call will be used. You can pass `.test(ckpt_path='best')` to use the best model or `.test(ckpt_path='last')` to use the last model. If you pass a value, this warning will be silenced.
/usr/local/lib/python3.10/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      testBinaryAUROC      │    0.8416043519973755     │
│     testBinaryF1Score     │     0.714112401008606     │
└───────────────────────────┴───────────────────────────┘

[{'testBinaryAUROC': 0.8416043519973755,
  'testBinaryF1Score': 0.714112401008606}]

In [176]:
test = torch.tensor(pd.read_csv('/kaggle/input/aith-dl-competition-tabular-data/test.csv').drop(columns = ['id']).values).float()
data_test_loader = DataLoader(test, batch_size = 1024)
preds = trainer.predict(model = model, dataloaders = data_test_loader)
all_preds = torch.cat(preds)
all_preds = F.sigmoid(all_preds)

submission = pd.DataFrame({
    "id": range(15000, len(all_preds) + 15000), 
    "smoking": all_preds.numpy(),
})

submission.to_csv("submission.csv", index = False)

/usr/local/lib/python3.10/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]